# Reinforcement Learning Lab Sheet  
## Hierarchical Reinforcement Learning with Options

In this lab, you will compare the performance tabular **Q-Learning** agent that uses only primitive actions to a **Macro-Q Learning** agent that uses both primitive actions and options to explore and learn about its environment.

## Environment

We provide a deterministic gridworld environment implemented in the `GridWorldEnv` class.

- **State**: a tuple `(y, x)` of **integers** representing the agent's current position (row, column).
  - Throughout the lab, be careful to remain consistent with the `(y, x)` convention.
- **Actions**: The agent has access to four actions corresponding to movement in the cardinal directions:
  - `0`: East, `1`: South, `2`: West, `3`: North.
- **Dynamics**: Actions move the agent deterministically in the intended direction. Attempting to step into a wall leaves the agent in the same state.
- **Rewards**: The agent receives a small step cost for each action taken ($-0.01$), and a positive reward upon reaching the goal state ($+1.0$).
- **Initiation and termination**: Each episode starts in a fixed initial state and terminates when the agent reaches the goal state.

The `GridWorldEnv` class provides the following methods and attributes:

- `.reset()` returns the initial state.
- `.step(a)` executes action `a` and returns a tuple `(s', r, done)` consisting of:
  - the next state (`State`),
  - the reward (`float`),
  - and a `boolean` indicating whether the episode has terminated.
- `.render()` displays a visualisation of the gridworld and the agent's current position:
  - the initial state is shown in green,
  - the agent's current state is shown in blue,
  - the goal state is shown in red,
  - walls are shown in black, and walkable floor tiles in white.
- `.num_actions` gives the number of available actions (always $4$).
- `.height` and `.width` give the height and width of the gridworld.

The following example code initialises the environment object, resets it to an initial state, performs one step of interaction, and visualises the results.

In [4]:
from lab_materials_hrl import GridWorldEnv

# Initialise the environment object.
env = GridWorldEnv()

# Reset the environment to an initial state.
state = env.reset()

# Render the initial state.
%matplotlib inline
env.render()
print(f"Initial State: {state}")

ModuleNotFoundError: No module named 'lab_materials_hrl'

In [3]:
# Go South one step.
action = 1
next_state, reward, terminal = env.step(action)

# Render environment and print transition information.
env.render()
print(f"(State: {state}, Action: {action}) --> (Reward: {reward}, Next-State: {next_state})")

NameError: name 'env' is not defined

## Basic Q-Learning

Below, we have provided you with a class implementing a standard **tabular Q-Learning** agent.

### What to look for in the code below.

- Action-value estimates are stored in a dictionary `q_table[(state, action)]`.
- Actions are chosen using an **ε-greedy** exploration strategy:
  - with probability `epsilon`, choose a random action;
  - otherwise choose an action with maximal value (with ties being broken uniformly at random).
- The update is the usual one-step Q-Learning target:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha \left( r + \gamma \max_{a'} Q(s',a') - Q(s,a) \right)
$$

Read the implementation below carefully: you will extend it to develop your Macro-Q Learning agent.


In [ ]:
import random
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List, Union

# The state returned by reset/step is a tuple of ints (y, x).
State = Tuple[int, int]  # (y, x)

# Actions to be provided as input to step are ints.
Action = int


# ============================================================
# Agents
# ============================================================


@dataclass
class AgentConfig:
    alpha: float = 0.1
    gamma: float = 1.0
    epsilon: float = 0.1


class BaseAgent:
    def __init__(self, cfg: AgentConfig, num_actions: int):
        self.cfg = cfg
        self.num_actions = num_actions
        self.gamma = cfg.gamma
        self.q_table: Dict[Tuple[State, object], float] = {}

    def get_q(self, state: State, action: Action) -> float:
        return self.q_table.get((state, action), 0.0)

    def set_q(self, state: State, action: Action, value: float) -> None:
        self.q_table[(state, action)] = value

    def select_action(self, state: State) -> Action:
        raise NotImplementedError()

    def greedy_action(self, state: State) -> Action:
        raise NotImplementedError()

    def reset_agent_state(self) -> State:
        raise NotImplementedError()


class PrimitiveQLearningAgent(BaseAgent):
    """
    Standard tabular Q-learning with dict-backed Q-table.

    q_table[(s,a)] = Q(s,a)
    """

    def select_action(self, state: State) -> Action:
        if random.random() < self.cfg.epsilon:
            return random.randrange(self.num_actions)
        return self.greedy_action(state)

    def greedy_action(self, state: State) -> Action:
        q_vals = [self.get_q(state, action) for action in range(self.num_actions)]
        max_q = max(q_vals)
        best_actions = [action for action, q_val in enumerate(q_vals) if q_val == max_q]
        return random.choice(best_actions)

    def reset_agent_state(self) -> None:
        return

    def train_step(self, env: GridWorldEnv, state: State) -> State:
        # Select action.
        action = self.select_action(state)

        # Execute action, observe next-state and reward.
        next_state, reward, terminal = env.step(action)

        # Perform one-step Q-Learning update.
        q_sa = self.get_q(state, action)
        bootstrap = (
            0.0 if terminal else max(self.get_q(next_state, next_action) for next_action in range(self.num_actions))
        )
        target = reward + self.cfg.gamma * bootstrap

        self.set_q(state, action, q_sa + self.cfg.alpha * (target - q_sa))

        if not terminal:
            return next_state
        else:
            self.reset_agent_state()
            return env.reset()


## Options

An **option** is a temporally extended action. Each option consists of:

- an **initiation set**: states where the option can be started,
- an internal **policy** over primitive actions,
- a **termination condition**.

We have provided a `SubgoalOption` class with methods corresponding to each of these elements:

- `.initiation(s)` returns a `bool` indicating whether the option can be initiated in state `s`.
- `.policy(s)` returns an `Action` (`int`) corresponding to the primitive action the option would execute in state `s`.
- `.termination(s)` returns a `float` indicating the probability that the option terminates in state `s`.
  - In this lab sheet, this will always be either `0.0` or `1.0`.

You construct a `SubgoalOption` by specifying:

- a reference to the agent's **environment**,
- a **subgoal** state `(y, x)`, and
- an **initiation set size** $k$.

The option's initiation set contains the nearest $k$ walkable states to the subgoal, where "nearest" is measured by **shortest-path distance** in the gridworld. The option's policy is computed automatically using value iteration, and aims to take the agent efficiently to the subgoal. The option terminates when the agent either reaches the subgoal or leaves the initiation set.

For instance, an option that allows an agent to navigate to the doorway between the top-left and top-right doorway (grid coordinate $(3, 7)$) from the nearest $40$ states can be constructed using the following code: `option = SubgoalOption(env, subgoal=(3, 7), initiation_set_size=40)`

Finally, note that options implement `__hash__`, so they can be used as dictionary keys. This should prove useful below, where your agent will learn values for both primitive actions and options.


## Macro-Q Learning

When the agent selects an option $o$ in state $s_t$, the option executes for $k$ time steps, producing rewards $r_{t+1}, r_{t+2}, \ldots, r_{t+k}$, and terminates in state $s_{t+k}$.

Whenever an option terminates, Macro-Q Learning performs the following update:

$$
Q(s_t, o) \leftarrow Q(s_t, o) + \alpha\left[
r_{t+1} + \gamma r_{t+2} + \ldots + \gamma^{k-1} r_{t+k} + \gamma^{k} \max_{oc'} Q(s_{t+k}, o') - Q(s_t, o)
\right],
$$

where $o'$ is the highest-valued primitive action or option available in state $s_{t+k}$.

Note that for primitive actions this reduces to a one-step Q-Learning update with $k=1$.

Your task is to implement a **Macro-Q Learning agent** that can act using both primitive actions and options. You should do this by completing the missing methods in the `MacroQLearningAgent` class provided below (a subclass of `BaseAgent`, which `QLearningAgent` also subclasses).

As a guide, your agent will need to

- learn the value of both primitive actions and options,
- keep track of whether an option is currently executing,
- follow the option's internal policy while it is executing,
- perform a Macro-Q update when an option terminates, and
- choose a new primitive action or option whenever the current option terminates.


In [1]:
from lab_materials_hrl import Option, SubgoalOption

PrimitiveOrOption = Union[Action, Option]


class MacroQLearningAgent(BaseAgent):
    """
    Agent that can choose among primitive actions and options.

    Key idea:
      - When the agent is *not* currently executing an option, it chooses either a primitive action
        or an initiable option (ε-greedy over the combined set).
      - When the agent *is* executing an option, it follows the option's internal policy until
        the option terminates (or the environment terminates).
      - When an option terminates after k steps, we perform a Macro-Q update on Q(s_start, option).
    """

    def __init__(self, cfg: AgentConfig, env: GridWorldEnv, options: List[Option]):
        super().__init__(cfg, num_actions=env.num_actions)
        self.options = options

        # Variables to track information about the option currently being executed.
        self.current_option: Optional[Option] = None
        self.option_start_state: Optional[State] = None
        self.option_return: float = 0.0
        self.option_discount: float = 1.0
        self.option_duration: int = 0

    def reset_agent_state(self) -> None:
        self.current_option = None
        self.option_start_state = None
        self.option_return = 0.0
        self.option_discount = 1.0
        self.option_duration = 0

    def _get_available_choices(self, state: State) -> List[PrimitiveOrOption]:
        choices: List[PrimitiveOrOption] = list(range(self.num_actions))
        for option in self.options:
            if option.initiation(state):
                choices.append(option)
        return choices

    def select_action(self, state: State) -> PrimitiveOrOption:
        # If we are currently following an option, follow its option policy.
        if self.current_option is not None:
            return self.current_option.policy(state)
        # Otherwise, choose epsilon-greedily over primitive actions and options.
        else:
            if random.random() < self.cfg.epsilon:
                return random.choice(self._get_available_choices(state))
            else:
                return self.greedy_action(state)

    def greedy_action(self, state: State) -> PrimitiveOrOption:
        # Greedy choice over primitive actions and options.
        choices = self._get_available_choices(state)
        q_vals = [self.get_q(state, choice) for choice in choices]
        max_q = max(q_vals)
        best_choices = [choice for choice, q_val in zip(choices, q_vals) if q_val == max_q]
        return random.choice(best_choices)

    def train_step(self, env: GridWorldEnv, state: State) -> State:
        # # # # # # # # # # # # # # # # # # # # # # # # # # #
        # Case 1: We are not currently executing an option. #
        # # # # # # # # # # # # # # # # # # # # # # # # # # #
        if self.current_option is None:
            choice = self.select_action(state)

            # Chosen primitive action: execute action, observe next state and reward, perform one-step update (k=1).
            if isinstance(choice, int):
                action = int(choice)
                next_state, reward, terminal = env.step(action)

                q_sa = self.get_q(state, action)
                bootstrap = (
                    0.0
                    if terminal
                    else max(
                        self.get_q(next_state, next_choice) for next_choice in self._get_available_choices(next_state)
                    )
                )
                target = reward + self.cfg.gamma * bootstrap
                self.set_q(state, action, q_sa + self.cfg.alpha * (target - q_sa))

                if not terminal:
                    return next_state
                else:
                    self.reset_agent_state()
                    return env.reset()

            # Chosen option: set option as current option and start executing it.
            o = choice
            self.current_option = o
            self.option_start_state = state
            self.option_return = 0.0
            self.option_discount = 1.0
            self.option_duration = 0

        # # # # # # # # # # # # # # # # # # # # # # # # #
        # Case 2: We are currently executing an option. #
        # # # # # # # # # # # # # # # # # # # # # # # # #
        assert self.current_option is not None
        assert self.option_start_state is not None

        action = self.select_action(state)
        next_state, reward, terminal = env.step(action)

        # Accumulate discounted reward during option execution
        self.option_return += self.option_discount * float(reward)
        self.option_discount *= self.cfg.gamma
        self.option_duration += 1

        option_terminated = self.current_option.termination(next_state)
        if terminal or option_terminated:
            # Macro-Q update for the option
            q_old = self.get_q(self.option_start_state, self.current_option)

            if terminal:
                target = self.option_return
            else:
                bootstrap = max(
                    self.get_q(next_state, next_choice) for next_choice in self._get_available_choices(next_state)
                )
                target = self.option_return + (self.cfg.gamma**self.option_duration) * bootstrap

            self.set_q(self.option_start_state, self.current_option, q_old + self.cfg.alpha * (target - q_old))
            # Clear option execution state.
            self.current_option = None
            self.option_start_state = None
            self.option_return = 0.0
            self.option_discount = 1.0
            self.option_duration = 0

        if not terminal:
            return next_state
        else:
            self.reset_agent_state()
            return env.reset()

ModuleNotFoundError: No module named 'lab_materials_hrl'

## Evaluation

We will evaluate both agents by plotting learning curves showing **evaluation performance**.

Specifically, the Q-Learning and Macro-Q Learning agents will be evaluated by:

1. Training them for a fixed number of time steps.
2. Periodically, every `EVAL_EVERY` time steps, the agent's current learned **greedy** policy (i.e. no ε-exploration) will be evaluated in a separate copy of the environment.
   - Each evaluation consists of running the agent for `EVAL_EPISODES` episodes.
   - If the agent selects an option during evaluation, it will continue executing that option until it terminates.
   - For safety, evaluation episodes are truncated after a large number of steps to prevent infinite loops if an agent gets stuck. (This does not affect training.)
3. The episodic returns produced over the course of training will be used to plot learning curves.

This process will be repeated `NUM_RUNS` times. The resulting learning curves will show mean episodic return, with shaded regions indicating a 95% confidence interval.

You do not need to interact with or modify the evaluation code yourself. However, it is provided in `lab_materials_hrl.py` for your reference.

First, let us define some `SubgoalOption`s for your Macro-Q Learning agent to use. The provided code defines four options corresponding to navigating to each of the four doorways from the nearest 40 states. You are encouraged to experiment with different subgoals and initiation set sizes to see how this impacts the results.


In [ ]:
from lab_materials_hrl import plot_grid_with_coordinates

# Plot a version of the grid with coordinates printed on each cell to help you decide which subgoals to use.
plot_grid_with_coordinates(env)

# Options corresponding to the doorways in the four-room environment (you can experiment with changing these later).
options = [
    SubgoalOption(env, subgoal=(3, 7), initiation_set_size=40),
    SubgoalOption(env, subgoal=(9, 6), initiation_set_size=40),
    SubgoalOption(env, subgoal=(6, 3), initiation_set_size=40),
    SubgoalOption(env, subgoal=(6, 10), initiation_set_size=40),
]


In [ ]:
from lab_materials_hrl import run_experiment, plot_learning_curves, set_global_seeds


TOTAL_STEPS = 10_000
EVAL_EVERY = 100
EVAL_EPISODES = 10
NUM_RUNS = 10


def make_env() -> GridWorldEnv:
    return GridWorldEnv()


def make_primitive_agent(env: GridWorldEnv) -> PrimitiveQLearningAgent:
    cfg = AgentConfig(alpha=0.2, gamma=1.0, epsilon=0.1)
    return PrimitiveQLearningAgent(cfg, num_actions=env.num_actions)


def make_option_agent(env: GridWorldEnv) -> MacroQLearningAgent:
    cfg = AgentConfig(alpha=0.2, gamma=1.0, epsilon=0.1)

    return MacroQLearningAgent(cfg, env, options=options)


set_global_seeds(0)
primitive_traces = run_experiment(
    make_env,
    make_primitive_agent,
    total_steps=TOTAL_STEPS,
    eval_every=EVAL_EVERY,
    eval_episodes=EVAL_EPISODES,
    num_runs=NUM_RUNS,
    base_seed=0,
)

set_global_seeds(0)
option_traces = run_experiment(
    make_env,
    make_option_agent,
    total_steps=TOTAL_STEPS,
    eval_every=EVAL_EVERY,
    eval_episodes=EVAL_EPISODES,
    num_runs=NUM_RUNS,
    base_seed=0,
)

plot_learning_curves(
    {
        "Q-Learning": primitive_traces,
        "Macro-Q Learning": option_traces,
    },
)


## Things to Try Next

Once your Macro-Q Learning agent is working, you should typically observe that options corresponding to the doorway states allow it to learn more rapidly than the primitive Q-Learning agent.

You may now wish to experiment by

- changing the subgoal states your options navigate to,
- varying the size of the initiation sets, or
- altering the number of options available to the agent.

How sensitive is performance to these design choices? Are all subgoal choices useful? Do these changes yield any surprising results?
